# Privacy-Aware Android App Permission Management
## Borda-Garrett Ranking + Decision-Tree IF-THEN Rules + Reinforcement Learning

**Category focus:** `Games`

## 2. Imports and Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import os
import json
import random
import itertools

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import norm

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier, _tree
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2, mutual_info_classif
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
    roc_auc_score, average_precision_score, classification_report, confusion_matrix
)

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
print("LightGBM available:", HAS_LGB)

In [ ]:
# ---------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# ---------------------------------------------------------------
# Project configuration
# ---------------------------------------------------------------
CATEGORY = "Games"
# The raw `Category` column has no literal "Games" value. It has six game genres, which are
# mapped explicitly onto the broad project category (Section 6). Nothing is fabricated.
CATEGORY_RAW_VALUES = ["Arcade & Action", "Brain & Puzzle", "Casual",
                       "Cards & Casino", "Racing", "Sports Games"]

DATA_PATH = "Android_Permission.csv"          # pandas reads the .gz directly
WEIGHTS_PATH = "combined_weights_games.csv"   # written in Section 9 from TRAIN data only
REFERENCE_WEIGHTS_PATH = "combined_weights_social_communication.csv"  # optional, comparison only
OUTPUT_DIR = "results_RL_Games"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TOP_K = 12

METADATA_FEATURES = ["Number of ratings", "Dangerous permissions count", "Safe permissions count"]

MIN_RAW_PERMISSION_SUPPORT = 0.005   # Games has ~4.9k training apps, so 0.5% = ~25 apps
SMOTE_TARGET_RATIO = 1.0
N_CV_FOLDS = 5

# ---- Precision focus --------------------------------------------------------
# Final threshold = most accurate threshold on VALIDATION whose bootstrap-lower-bound precision >= PRECISION_TARGET.
PRECISION_TARGET = 0.85
FALLBACK_BETA = 0.5          # F-beta used if no threshold reaches the target (beta<1 favours precision)
MIN_RECALL_FLOOR = 0.40      # guard so the constraint can't be met by flagging only a handful of apps
THRESH_N_BOOT = 500          # bootstrap resamples of VALIDATION used to get a conservative precision estimate
THRESH_PRECISION_QUANTILE = 0.10   # the 10th-percentile precision must clear PRECISION_TARGET


COEF_CANDIDATES = [
    (0.40, 0.20, 0.10),
    (0.50, 0.20, 0.15),
    (0.60, 0.25, 0.15),
    (0.70, 0.20, 0.10),
    (0.80, 0.30, 0.20),
    (0.90, 0.35, 0.25),
    (0.60, 0.40, 0.20),
]

# Q-learning
QL_ALPHA = 0.1
QL_GAMMA = 0.9
QL_PASSES = 3                # episodes = QL_PASSES * len(balanced train set)
QL_EPS_START = 1.0
QL_EPS_MIN = 0.05
QL_EPS_DECAY = 0.9995

PRIVACY_LOW_THRESHOLD = 0.33
PRIVACY_HIGH_THRESHOLD = 0.67

print("CATEGORY:", CATEGORY, "<- raw Category values:", CATEGORY_RAW_VALUES)
print("TOP_K:", TOP_K, "| PRECISION_TARGET:", PRECISION_TARGET)
print("Reward coefficient candidates:", COEF_CANDIDATES)

## 3. Load Dataset

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
print("Dataset shape:", df.shape)
missing_counts = df.isna().sum()
print("\nMissing values (columns with >0):")
print(missing_counts[missing_counts > 0])
print("\nClass distribution (0 = Benign, 1 = Malicious):")
print(df["Class"].value_counts())
print("\nGame-genre rows in the raw data:")
print(df.loc[df["Category"].isin(CATEGORY_RAW_VALUES), "Category"].value_counts())

## 5. Data Validation

Same 12-permission aggregation map as the Social & Communication notebook, so results are comparable
across categories. Missing columns are reported, never fabricated.

In [ ]:
PERMISSION_COLUMN_MAP = {
    "Camera": ["Hardware controls : take pictures and videos (D)"],
    "Location": [
        "Your location : coarse (network-based) location (D)",
        "Your location : fine (GPS) location (D)",
        "Default : coarse (network-based) location (S)",
    ],
    "Contacts": [
        "Your personal information : read contact data (D)",
        "Your personal information : write contact data (D)",
        "Default : write contact data (S)",
        "Your accounts : contacts data in Google accounts (D)",
    ],
    "Microphone": ["Hardware controls : record audio (D)"],
    "Storage": [
        "Storage : modify/delete USB storage contents modify/delete SD card contents (D)",
        "Default : modify/delete USB storage contents modify/delete SD card contents (S)",
    ],
    "SMS": [
        "Services that cost you money : send SMS messages (D)",
        "Your messages : edit SMS or MMS (D)",
        "Your messages : read SMS or MMS (D)",
        "Your messages : receive SMS (D)",
        "Your messages : receive MMS (D)",
        "Your messages : send SMS-received broadcast (S)",
    ],
    "CallLogs": [],
    "Bluetooth": [
        "Network communication : create Bluetooth connections (D)",
        "System tools : bluetooth administration (D)",
    ],
    "Calendar": [
        "Your personal information : add or modify calendar events and send email to guests (D)",
        "Your personal information : read calendar events (D)",
    ],
    "PhoneState": [
        "Default : read phone state and identity (S)",
        "Phone calls : read phone state and identity (D)",
    ],
    "Internet": [
        "Default : full Internet access (S)",
        "Network communication : full Internet access (D)",
    ],
    "ReadPhoneNumbers": [],
}
PERMISSION_NAMES = list(PERMISSION_COLUMN_MAP.keys())

missing_report = {k: [c for c in v if c not in df.columns] for k, v in PERMISSION_COLUMN_MAP.items()}
missing_report = {k: v for k, v in missing_report.items() if v}
print("Missing mapped columns:", missing_report if missing_report else "none")
print("Permissions with NO raw column in this dataset (constant 0):",
      [p for p, c in PERMISSION_COLUMN_MAP.items() if len(c) == 0])

## 6. Category Filtering

In [ ]:
df["BroadCategory"] = np.where(df["Category"].isin(CATEGORY_RAW_VALUES), CATEGORY, "Other")
print(df["BroadCategory"].value_counts())

cat_df = df[df["BroadCategory"] == CATEGORY].copy().reset_index(drop=True)
print(f"\nApps in category '{CATEGORY}':", len(cat_df))
print("Benign  (Class=0):", (cat_df["Class"] == 0).sum())
print("Malicious (Class=1):", (cat_df["Class"] == 1).sum())
print("\nMalicious rate by genre:")
print(cat_df.groupby("Category")["Class"].agg(["count", "mean"]).round(3))

## 7. Permission Matrix + Feature Engineering

12 aggregated permission flags (RL-visible), raw permission columns (classifier-only), metadata,
and engineered features. All engineered features are derived from existing columns.

In [ ]:
def build_permission_matrix(source_df, colmap):
    out = {}
    for perm, cols in colmap.items():
        present = [c for c in cols if c in source_df.columns]
        out[perm] = ((source_df[present].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) > 0)
                     .astype(int) if present else pd.Series(0, index=source_df.index))
    return pd.DataFrame(out)

perm_feats = build_permission_matrix(cat_df, PERMISSION_COLUMN_MAP)

parts = {
    "App": cat_df["App"].fillna("").values,
    "Category": cat_df["Category"].values
}

# Add permission-level aggregated features
for p in PERMISSION_NAMES:
    parts[p] = perm_feats[p].values

# Add ONLY original metadata features here
BASE_METADATA_FEATURES = [
    "Dangerous permissions count",
    "Safe permissions count",
    "Number of ratings"
]

for m in BASE_METADATA_FEATURES:
    if m in cat_df.columns:
        parts[m] = pd.to_numeric(
            cat_df[m], errors="coerce"
        ).fillna(0).values

permission_matrix = pd.DataFrame(parts)

# --- Engineered metadata ---------------------------------------------------------------
permission_matrix["PermissionCount"] = permission_matrix[PERMISSION_NAMES].sum(axis=1)
_d = permission_matrix["Dangerous permissions count"].astype(float)
_s = permission_matrix["Safe permissions count"].astype(float)
permission_matrix["DangerousToSafeRatio"] = (_d / (_d + _s + 1e-6)).clip(0, 1)
permission_matrix["LogNumRatings"] = np.log1p(permission_matrix["Number of ratings"].clip(lower=0))
permission_matrix["DescriptionLength"] = cat_df["Description"].fillna("").str.len().values

GENRE_FEATURES = [f"Genre_{g}" for g in CATEGORY_RAW_VALUES]
for g, col in zip(CATEGORY_RAW_VALUES, GENRE_FEATURES):
    permission_matrix[col] = (permission_matrix["Category"] == g).astype(int)

ENGINEERED_METADATA_FEATURES = ["PermissionCount", "DangerousToSafeRatio", "LogNumRatings", "DescriptionLength"]
METADATA_FEATURES = METADATA_FEATURES + ENGINEERED_METADATA_FEATURES + GENRE_FEATURES

# --- Raw permission columns for the classifier -----------------------------------------
NON_PERMISSION_COLUMNS = {
    "App", "Package", "Category", "BroadCategory", "Description", "Related apps",
    "Rating", "Number of ratings", "Price",
    "Dangerous permissions count", "Safe permissions count", "Class",
}
RAW_PERMISSION_COLUMNS_ALL = [c for c in df.columns if c not in NON_PERMISSION_COLUMNS]
raw_block = cat_df[RAW_PERMISSION_COLUMNS_ALL].apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)
permission_matrix = pd.concat([permission_matrix, raw_block.reset_index(drop=True)], axis=1)

# Text used for DescriptionTextRisk (Section 11b). Kept in a side array, not a model feature itself.
APP_TEXT = (cat_df["App"].fillna("") + " . " + cat_df["Description"].fillna("")).values

permission_matrix["LabelEncoded"] = cat_df["Class"].astype(int).values
permission_matrix["uid"] = np.arange(len(permission_matrix))

RAW_TO_AGG = {c: perm for perm, cols in PERMISSION_COLUMN_MAP.items() for c in cols}

print("permission_matrix shape:", permission_matrix.shape)
print("\nPermission support (fraction of Games apps):")
print(permission_matrix[PERMISSION_NAMES].mean().round(4).sort_values(ascending=False))

## 8. Train / Validation / Test Split (70 / 15 / 15, Stratified)

Happens **before** weight computation, rule mining, text modelling, classifier training and RL training.
Stratified on the label *and* genre so every genre is represented proportionally in each split.

In [ ]:
strat_key = permission_matrix["LabelEncoded"].astype(str) + "_" + permission_matrix["Category"]
train_df, temp_df = train_test_split(permission_matrix, test_size=0.30, stratify=strat_key,
                                     random_state=RANDOM_STATE)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=strat_key.loc[temp_df.index],
                                   random_state=RANDOM_STATE)
train_df, val_df, test_df = train_df.copy(), val_df.copy(), test_df.copy()

for n, d in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"{n:5s}: {d.shape}  malicious rate: {d['LabelEncoded'].mean():.3f}")

assert set(train_df.index).isdisjoint(val_df.index)
assert set(train_df.index).isdisjoint(test_df.index)
assert set(val_df.index).isdisjoint(test_df.index)
print("\nVerified: train / validation / test indices are mutually disjoint.")

# Classifier raw-permission features: support measured on TRAIN only
_raw_support = train_df[RAW_PERMISSION_COLUMNS_ALL].mean()
CLF_PERMISSION_FEATURES = [c for c in RAW_PERMISSION_COLUMNS_ALL
                           if train_df[c].nunique() > 1 and _raw_support[c] >= MIN_RAW_PERMISSION_SUPPORT]
print(f"\nRaw permission columns kept for classifier (support >= {MIN_RAW_PERMISSION_SUPPORT} on TRAIN):",
      len(CLF_PERMISSION_FEATURES))


### 9b. Load & validate the Games weights file

In [ ]:
weights_df = pd.read_csv(WEIGHTS_PATH)
weight_sum = weights_df["CombinedWeight"].sum()
print("Sum of CombinedWeight:", weight_sum)
if not np.isclose(weight_sum, 1.0, atol=1e-2):
    weights_df["CombinedWeight"] /= weight_sum
    print("Normalised to sum 1.")
weights_df = weights_df.sort_values("CombinedWeight", ascending=False).reset_index(drop=True)
weights_df["Rank"] = weights_df.index + 1
weight_map = dict(zip(weights_df["Permission"], weights_df["CombinedWeight"]))

if os.path.exists(REFERENCE_WEIGHTS_PATH):
    ref = pd.read_csv(REFERENCE_WEIGHTS_PATH).rename(columns={"CombinedWeight": "SocialComm_Weight"})
    weights_df = weights_df.merge(ref, on="Permission", how="left")
    print("Social & Communication weights shown for comparison only (not used).")
weights_df

## 10. Top Permission Selection (RL action space)

In [ ]:
top_permissions = weights_df.head(TOP_K)["Permission"].tolist()
RL_FEATURES = top_permissions
print(f"TOP_{TOP_K} permissions for the RL state/action space:", top_permissions)
weights_df.head(TOP_K)[["Rank", "Permission", "CombinedWeight"]]

## 11. Interpretable Decision-Tree Rules (TRAINING SET ONLY)

The tree uses only `train_df`, `RL_FEATURES`, and `LabelEncoded`. Fixed depth and leaf-size candidates are compared on validation average precision and balanced accuracy. The test set is not used for fitting or selection. Every displayed rule is an actual root-to-leaf path from the selected tree.

In [ ]:
# Fit candidate trees only on the original training split; no SMOTE rows or metadata enter rule fitting.
TREE_CANDIDATES = [(depth, leaf) for depth in range(2, 6) for leaf in (25, 50)]
X_rule_train = train_df[RL_FEATURES].astype(float)
y_rule_train = train_df["LabelEncoded"].astype(int)
X_rule_val = val_df[RL_FEATURES].astype(float)
y_rule_val = val_df["LabelEncoded"].astype(int)

tree_candidates = []
for depth, leaf in TREE_CANDIDATES:
    candidate = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=leaf,
                                       class_weight=None, random_state=RANDOM_STATE)
    candidate.fit(X_rule_train, y_rule_train)
    val_proba = candidate.predict_proba(X_rule_val)[:, list(candidate.classes_).index(1)]
    val_pred = (val_proba >= 0.5).astype(int)
    tree_candidates.append({
        "max_depth": depth, "min_samples_leaf": leaf,
        "tree": candidate,
        "Val_AvgPrecision": average_precision_score(y_rule_val, val_proba),
        "Val_BalancedAccuracy": (recall_score(y_rule_val, val_pred, zero_division=0)
                                 + recall_score(y_rule_val, val_pred, pos_label=0, zero_division=0)) / 2,
        "Leaves": candidate.get_n_leaves(),
    })
tree_selection_df = pd.DataFrame([{k:v for k,v in r.items() if k != "tree"} for r in tree_candidates])
best_ap = tree_selection_df["Val_AvgPrecision"].max()
# Favor the simplest candidate within 0.01 AP of the best; then higher balanced accuracy.
eligible = tree_selection_df[tree_selection_df["Val_AvgPrecision"] >= best_ap - 0.01]
chosen_idx = eligible.sort_values(["max_depth", "Leaves", "Val_BalancedAccuracy", "min_samples_leaf"],
                                  ascending=[True, True, False, False]).index[0]
RULE_TREE = tree_candidates[int(chosen_idx)]["tree"]
print("Decision-tree candidates (validation only):")
display(tree_selection_df.drop(columns=[], errors="ignore").round(4))
print(f"Selected shallowest tree within 0.01 validation AP of best (best AP={best_ap:.4f}).")

def extract_tree_rules(tree, feature_names):
    t = tree.tree_
    found = []
    def walk(node, conditions):
        if t.feature[node] != _tree.TREE_UNDEFINED:
            feature = feature_names[t.feature[node]]
            threshold = t.threshold[node]
            # Inputs are binary, so <= 0.5 is NOT permission and > 0.5 is permission.
            walk(t.children_left[node], conditions + [(feature, 0)])
            walk(t.children_right[node], conditions + [(feature, 1)])
            return
        # Store only the path here. Class counts and prediction are recomputed from
        # the original train_df rows below, avoiding sklearn tree-value normalization
        # differences across versions.
        found.append({"Conditions": tuple(conditions)})
    walk(0, [])
    return found

def format_conditions(conditions):
    if not conditions:
        return "(no conditions)"
    return " AND ".join((f"{name}" if value == 1 else f"NOT {name}") for name, value in conditions)

_rule_paths = extract_tree_rules(RULE_TREE, RL_FEATURES)
_train_leaf_ids = RULE_TREE.apply(X_rule_train)
_val_leaf_ids = RULE_TREE.apply(X_rule_val)
_tree_leaf_nodes = [i for i in range(RULE_TREE.tree_.node_count) if RULE_TREE.tree_.children_left[i] == _tree.TREE_LEAF]
_node_to_rule = {node: i + 1 for i, node in enumerate(_tree_leaf_nodes)}
rule_rows = []
for node, path in zip(_tree_leaf_nodes, _rule_paths):
    trmask, vamask = _train_leaf_ids == node, _val_leaf_ids == node
    path = dict(path)
    train_counts = y_rule_train.values[trmask]
    path["TrainingApps"] = int(trmask.sum())
    path["TrainingMalicious"] = int(train_counts.sum())
    path["TrainingBenign"] = int(trmask.sum() - train_counts.sum())
    path["MaliciousProbability"] = float(train_counts.mean()) if len(train_counts) else np.nan
    # sklearn predicts the majority class in each leaf; ties resolve to the lower class (0).
    path["PredictedClass"] = int(path["TrainingMalicious"] > path["TrainingBenign"])
    path.update({"RuleNumber": _node_to_rule[node], "ConditionsText": format_conditions(path["Conditions"]),
                 "ValidationSupport": int(vamask.sum()),
                 "ValidationMalicious": int(y_rule_val.values[vamask].sum()),
                 "ValidationBenign": int(vamask.sum() - y_rule_val.values[vamask].sum()),
                 "ValidationMaliciousRate": float(y_rule_val.values[vamask].mean()) if vamask.any() else np.nan,
                 "Node": node})
    rule_rows.append(path)
TREE_RULES = rule_rows
rules_df = pd.DataFrame([{k:v for k,v in row.items() if k not in ("Conditions", "Node")} for row in TREE_RULES])

tree_val_proba = RULE_TREE.predict_proba(X_rule_val)[:, list(RULE_TREE.classes_).index(1)]
tree_val_pred = (tree_val_proba >= 0.5).astype(int)
print("Selected tree validation performance:")
print({"Accuracy": accuracy_score(y_rule_val, tree_val_pred),
       "Precision": precision_score(y_rule_val, tree_val_pred, zero_division=0),
       "Recall": recall_score(y_rule_val, tree_val_pred, zero_division=0),
       "F1": f1_score(y_rule_val, tree_val_pred, zero_division=0),
       "AveragePrecision": average_precision_score(y_rule_val, tree_val_proba)})
print("Root-to-leaf IF-THEN rules; validation support is shown per leaf:")
display(rules_df[["RuleNumber", "ConditionsText", "PredictedClass", "MaliciousProbability", "TrainingApps",
                  "TrainingMalicious", "TrainingBenign", "ValidationSupport", "ValidationMalicious",
                  "ValidationBenign", "ValidationMaliciousRate"]].round(4))

# Only predicted-malicious leaf rules contribute to interaction risk.
MALICIOUS_TREE_RULES = [r for r in TREE_RULES
                        if r["PredictedClass"] == 1 and r["Conditions"]
                        and np.isfinite(r["MaliciousProbability"])]
if not MALICIOUS_TREE_RULES:
    print("No defensible high-risk rule: no non-empty malicious-predicted leaf path with a finite probability; interaction penalty is zero.")
elif not any(r["MaliciousProbability"] >= 0.5 and r["TrainingApps"] > 0 for r in MALICIOUS_TREE_RULES):
    print("No defensible high-risk rule: no malicious-predicted leaf has at least 0.5 malicious probability.")
else:
    print("Malicious leaf rules available to RL:", [r["RuleNumber"] for r in MALICIOUS_TREE_RULES])
rules_df.to_csv(os.path.join(OUTPUT_DIR, "decision_tree_rules.csv"), index=False)
with open(os.path.join(OUTPUT_DIR, "decision_tree_rules.json"), "w", encoding="utf-8") as f:
    json.dump([{k:(list(v) if k == "Conditions" else v) for k,v in r.items()} for r in TREE_RULES], f, indent=2)
print("Saved extracted rules to", OUTPUT_DIR)


## 11b. Description Text Risk (TRAINING SET ONLY, out-of-fold)

In [ ]:
TEXT_C = 4.0
tr_pos = train_df["uid"].values
va_pos = val_df["uid"].values
te_pos = test_df["uid"].values

def make_text_model():
    return (TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=60000, sublinear_tf=True),
            LogisticRegression(C=TEXT_C, max_iter=4000))

text_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof = np.zeros(len(tr_pos))
y_tr_text = train_df["LabelEncoded"].values
for fold_tr, fold_va in text_cv.split(tr_pos, y_tr_text):
    vec, lr = make_text_model()
    Xf = vec.fit_transform(APP_TEXT[tr_pos[fold_tr]])       # vocabulary fit inside the fold too
    lr.fit(Xf, y_tr_text[fold_tr])
    oof[fold_va] = lr.predict_proba(vec.transform(APP_TEXT[tr_pos[fold_va]]))[:, 1]

text_vec, text_lr = make_text_model()
text_lr.fit(text_vec.fit_transform(APP_TEXT[tr_pos]), y_tr_text)

train_df["DescriptionTextRisk"] = oof
val_df["DescriptionTextRisk"] = text_lr.predict_proba(text_vec.transform(APP_TEXT[va_pos]))[:, 1]
test_df["DescriptionTextRisk"] = text_lr.predict_proba(text_vec.transform(APP_TEXT[te_pos]))[:, 1]

METADATA_FEATURES = METADATA_FEATURES + ["DescriptionTextRisk"]
print(f"Out-of-fold TRAIN AUC of text risk: {roc_auc_score(y_tr_text, oof):.4f}")
print(f"VALIDATION AUC of text risk:        {roc_auc_score(val_df['LabelEncoded'], val_df['DescriptionTextRisk']):.4f}")

## 12. SMOTE Balancing of the Training Set (RL training + utility estimation)

Used for RL training and utility estimation only (the classifier does its own SMOTE inside each CV fold).
Binary columns are rounded back to {0, 1}. Val/test are never touched.

In [ ]:
BINARY_COLS = sorted(set(PERMISSION_NAMES) | set(CLF_PERMISSION_FEATURES) | set(GENRE_FEATURES))
smote_input_cols = [
    c for c in (
        BINARY_COLS
        + [m for m in METADATA_FEATURES if m not in GENRE_FEATURES]
    )
    if c in train_df.columns
]
X_train_raw = train_df[smote_input_cols].values.astype(float)
y_train_raw = train_df["LabelEncoded"].values

vc = pd.Series(y_train_raw).value_counts()
print("Before SMOTE:", vc.to_dict())
target = int(vc.max() * SMOTE_TARGET_RATIO)
if target > vc.min():
    X_res, y_res = SMOTE(random_state=RANDOM_STATE, sampling_strategy={vc.idxmin(): target}).fit_resample(X_train_raw, y_train_raw)
else:
    X_res, y_res = X_train_raw, y_train_raw

train_df_balanced = pd.DataFrame(X_res, columns=smote_input_cols)
train_df_balanced["LabelEncoded"] = y_res
for c in BINARY_COLS:
    train_df_balanced[c] = (train_df_balanced[c] >= 0.5).astype(int)
n_syn = len(train_df_balanced) - len(train_df)
train_df_balanced["App"] = list(train_df["App"].values) + [f"SMOTE_synthetic_{i}" for i in range(n_syn)]
print("After SMOTE:", train_df_balanced["LabelEncoded"].value_counts().to_dict(), f"({n_syn} synthetic rows)")

## 13. Malicious-Risk Classifier (TRAINING SET ONLY, CV-selected by average precision)

Candidates are compared by **5-fold cross-validated average precision** on TRAIN (SMOTE inside each fold).
Validation metrics are reported only as an independent check; the test set plays no role here.

In [ ]:
CLF_FEATURES = CLF_PERMISSION_FEATURES + METADATA_FEATURES
print(f"Classifier features: {len(CLF_PERMISSION_FEATURES)} raw permissions + {len(METADATA_FEATURES)} metadata = {len(CLF_FEATURES)}")

X_trainfull, y_trainfull = train_df[CLF_FEATURES].values.astype(float), train_df["LabelEncoded"].values
X_val, y_val = val_df[CLF_FEATURES].values.astype(float), val_df["LabelEncoded"].values

cv = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

candidate_pipelines = {
    "RandomForest": (ImbPipeline([("smote", SMOTE(random_state=RANDOM_STATE)),
                                  ("clf", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))]),
                     {"clf__n_estimators": [500], "clf__max_depth": [None, 16], "clf__min_samples_leaf": [1, 3],
                      "clf__max_features": ["sqrt", 0.3]}),
    "HistGradientBoosting": (ImbPipeline([("smote", SMOTE(random_state=RANDOM_STATE)),
                                          ("clf", HistGradientBoostingClassifier(random_state=RANDOM_STATE))]),
                             {"clf__max_iter": [300, 600], "clf__learning_rate": [0.03, 0.06],
                              "clf__max_leaf_nodes": [15, 31], "clf__l2_regularization": [0.0, 1.0]}),
    "LogisticRegression": (ImbPipeline([("smote", SMOTE(random_state=RANDOM_STATE)),
                                        ("scale", StandardScaler()),
                                        ("clf", LogisticRegression(max_iter=5000))]),
                           {"clf__C": [0.03, 0.3, 3.0]}),
}
if HAS_LGB:
    candidate_pipelines["LightGBM"] = (
        ImbPipeline([("smote", SMOTE(random_state=RANDOM_STATE)),
                     ("clf", lgb.LGBMClassifier(random_state=RANDOM_STATE, verbose=-1, n_jobs=2))]),
        {"clf__n_estimators": [400, 800], "clf__learning_rate": [0.02, 0.05],
         "clf__num_leaves": [15, 31], "clf__subsample": [0.8], "clf__subsample_freq": [1],
         "clf__colsample_bytree": [0.7], "clf__min_child_samples": [20]})

risk_model_metrics, fitted_grids = [], {}
for name, (pipe, grid_params) in candidate_pipelines.items():
    grid = GridSearchCV(pipe, grid_params, scoring="average_precision", cv=cv, n_jobs=1, refit=True)
    grid.fit(X_trainfull, y_trainfull)
    fitted_grids[name] = grid
    vp = grid.predict_proba(X_val)[:, 1]
    vpred = (vp >= 0.5).astype(int)
    risk_model_metrics.append({
        "Model": name, "CV_AvgPrecision": grid.best_score_,
        "Val_AvgPrecision": average_precision_score(y_val, vp), "Val_AUC": roc_auc_score(y_val, vp),
        "Val_Accuracy@0.5": accuracy_score(y_val, vpred), "Val_Precision@0.5": precision_score(y_val, vpred),
        "Val_Recall@0.5": recall_score(y_val, vpred), "BestParams": grid.best_params_,
    })
    print(f"{name:22s} CV AP={grid.best_score_:.4f}  Val AP={risk_model_metrics[-1]['Val_AvgPrecision']:.4f}")

# ---- Soft-voting ensemble of the tuned candidates (evaluated with the SAME CV folds) ----------
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.base import clone
ensemble = VotingClassifier(
    estimators=[(n, clone(g.best_estimator_)) for n, g in fitted_grids.items()], voting="soft", n_jobs=1)
ens_cv_ap = cross_val_score(ensemble, X_trainfull, y_trainfull, cv=cv, scoring="average_precision").mean()
ensemble.fit(X_trainfull, y_trainfull)
vp = ensemble.predict_proba(X_val)[:, 1]; vpred = (vp >= 0.5).astype(int)
risk_model_metrics.append({
    "Model": "SoftVotingEnsemble", "CV_AvgPrecision": ens_cv_ap,
    "Val_AvgPrecision": average_precision_score(y_val, vp), "Val_AUC": roc_auc_score(y_val, vp),
    "Val_Accuracy@0.5": accuracy_score(y_val, vpred), "Val_Precision@0.5": precision_score(y_val, vpred),
    "Val_Recall@0.5": recall_score(y_val, vpred), "BestParams": "soft vote of tuned candidates"})
print(f"{'SoftVotingEnsemble':22s} CV AP={ens_cv_ap:.4f}  Val AP={risk_model_metrics[-1]['Val_AvgPrecision']:.4f}")

class _Holder:  # uniform access for the ensemble
    def __init__(self, est): self.best_estimator_ = est
fitted_grids["SoftVotingEnsemble"] = _Holder(ensemble)

risk_model_metrics_df = pd.DataFrame(risk_model_metrics).sort_values("CV_AvgPrecision", ascending=False).reset_index(drop=True)
display(risk_model_metrics_df)

BEST_RISK_MODEL_NAME = risk_model_metrics_df.iloc[0]["Model"]
risk_model = fitted_grids[BEST_RISK_MODEL_NAME].best_estimator_
print(f"Selected risk model (by CV average precision on TRAIN): {BEST_RISK_MODEL_NAME}")
print("Best hyperparameters:", risk_model_metrics_df.iloc[0]["BestParams"])

try:
    _est = risk_model if hasattr(risk_model, "named_steps") else fitted_grids["RandomForest"].best_estimator_
    imp = _est.named_steps["clf"].feature_importances_
    imp = imp / imp.sum()
    print("\nTop 15 features by importance (RandomForest component if an ensemble was selected):")
    for i in np.argsort(imp)[::-1][:15]:
        print(f"   {CLF_FEATURES[i][:72]:72s} {imp[i]:.4f}")
except (AttributeError, KeyError):
    pass

def predict_malicious_proba(X2d):
    return risk_model.predict_proba(np.asarray(X2d, dtype=float))[:, 1]

## 14. Evidence-Based Utility (TRAINING SET ONLY)

`Utility(perm) = P(perm | Benign) / max P(perm | Benign)` over the TOP_K permissions.

In [ ]:
benign_train = train_df_balanced[train_df_balanced["LabelEncoded"] == 0]
p_perm_given_benign = {p: float(benign_train[p].mean()) for p in RL_FEATURES}
util_max = max(p_perm_given_benign.values()) or 1.0
utility_map = {p: p_perm_given_benign[p] / util_max for p in RL_FEATURES}
utility_table = pd.DataFrame({
    "Permission": RL_FEATURES,
    "P_Present_Given_Benign": [p_perm_given_benign[p] for p in RL_FEATURES],
    "P_Present_Given_AllApps": [float(train_df_balanced[p].mean()) for p in RL_FEATURES],
    "UtilityNormalized": [utility_map[p] for p in RL_FEATURES],
}).sort_values("UtilityNormalized", ascending=False).reset_index(drop=True)
utility_table

## 15. Privacy Risk Model

| Component | Formula |
|---|---|
| `IndividualRisk(perm)` | `CombinedWeight(perm) / max CombinedWeight over TOP_K` |
| `ClassifierRisk(granted)` | `P(Malicious \| full permission context + metadata)`; denying an aggregate zeroes all its raw sub-columns |
| `InteractionRisk(granted, decided)` | maximum malicious probability among fully matched malicious decision-tree leaves; 0 if none match |

Classifier risk is **cached**: for each app, every subset of its requested TOP_K permissions is scored in one
batch. The cache key is the exact feature vector, so values are identical to calling the model per step.

In [ ]:
# ============================================================
# RL + Classifier Risk Utilities
# ============================================================

# ------------------------------------------------------------
# 1. Maximum RL permission weight
# ------------------------------------------------------------

MAX_WEIGHT = max(
    weight_map.get(p, 0.0)
    for p in RL_FEATURES
)

def individual_risk_normalized(perm):
    """
    Normalize the individual permission risk to [0, 1]
    using the maximum weight among RL_FEATURES.
    """
    if MAX_WEIGHT == 0:
        return 0.0

    return weight_map.get(perm, 0.0) / MAX_WEIGHT


# ------------------------------------------------------------
# 2. Map classifier-level permission features to aggregated
#    permission names
# ------------------------------------------------------------

_CLF_PERM_AGG = [
    RAW_TO_AGG.get(p)
    for p in CLF_PERMISSION_FEATURES
]


# ------------------------------------------------------------
# 3. Classifier prediction cache
# ------------------------------------------------------------

_CLS_CACHE = {}


# ------------------------------------------------------------
# 4. Build classifier feature vector
# ------------------------------------------------------------

def _feature_vector(granted, requested, metadata):

    perm_part = []

    for p, agg in zip(
        CLF_PERMISSION_FEATURES,
        _CLF_PERM_AGG
    ):

        if agg is not None and agg in RL_FEATURES:

            # Permission is considered available to classifier
            # only if it was requested AND granted.
            value = (
                requested.get(p, 0)
                if granted.get(agg, 0) == 1
                else 0
            )

        else:

            value = requested.get(p, 0)

        perm_part.append(float(value))

    return np.asarray(
        perm_part + list(metadata),
        dtype=float
    )


# ------------------------------------------------------------
# 5. Classifier risk
# ------------------------------------------------------------

def classifier_risk(
    granted,
    requested,
    metadata
):

    v = _feature_vector(
        granted,
        requested,
        metadata
    )

    key = v.tobytes()

    if key not in _CLS_CACHE:

        _CLS_CACHE[key] = float(
            predict_malicious_proba(
                v.reshape(1, -1)
            )[0]
        )

    return _CLS_CACHE[key]


# ------------------------------------------------------------
# 6. Interaction risk from fully matched malicious decision-tree paths

def _rule_matches(rule, granted, decided):
    """A path matches only when every condition is known and satisfied."""
    decided = set(decided)
    if not rule.get("Conditions"):
        return False
    for permission, required_value in rule["Conditions"]:
        if permission not in decided:
            return False
        if int(granted.get(permission, 0)) != required_value:
            return False
    return True

def interaction_risk(granted, decided_permissions):
    if granted is None or decided_permissions is None:
        raise ValueError("granted and decided_permissions are required")
    if not MALICIOUS_TREE_RULES:
        return 0.0, []
    matched = [r for r in MALICIOUS_TREE_RULES if _rule_matches(r, granted, decided_permissions)]
    if not matched:
        return 0.0, []
    # Explicit weighting: equal rule weights, aggregated by maximum leaf malicious probability.
    # At complete decisions leaves are mutually exclusive; max remains safe for partial matches.
    risk = max(float(r["MaliciousProbability"]) for r in matched)
    if not np.isfinite(risk):
        raise ValueError("Matched decision-tree rule has missing malicious probability")
    names = [f"Rule {r['RuleNumber']}: {r['ConditionsText']}" for r in matched]
    return float(np.clip(risk, 0.0, 1.0)), names


# 7. Requested permission vector
# ------------------------------------------------------------

def make_requested_vector(row):

    d = {}

    # Aggregated permission features
    for p in PERMISSION_NAMES:

        if p in row.index:

            value = row[p]

            # Handle duplicate DataFrame columns
            if isinstance(value, pd.Series):
                value = value.iloc[0]

            value = pd.to_numeric(
                value,
                errors="coerce"
            )

            d[p] = (
                0
                if pd.isna(value)
                else int(value)
            )

        else:

            d[p] = 0

    # Raw classifier permission features
    for p in CLF_PERMISSION_FEATURES:

        if p in row.index:

            value = row[p]

            # Handle duplicate DataFrame columns
            if isinstance(value, pd.Series):
                value = value.iloc[0]

            value = pd.to_numeric(
                value,
                errors="coerce"
            )

            d[p] = (
                0
                if pd.isna(value)
                else int(value)
            )

        else:

            d[p] = 0

    return d


# ------------------------------------------------------------
# 8. Metadata vector
# ------------------------------------------------------------

def make_metadata_vector(row):

    metadata = []

    for m in METADATA_FEATURES:

        # If the feature is completely absent
        if m not in row.index:

            # Temporary fallback so the current pipeline
            # does not crash.
            metadata.append(0.0)
            continue

        value = row[m]

        # IMPORTANT:
        # If the DataFrame contains duplicate column names,
        # row[m] becomes a Series instead of a scalar.
        if isinstance(value, pd.Series):

            # Take the first duplicate occurrence.
            value = value.iloc[0]

        value = pd.to_numeric(
            value,
            errors="coerce"
        )

        if pd.isna(value):

            value = 0.0

        metadata.append(
            float(value)
        )

    return np.asarray(
        metadata,
        dtype=float
    )


# ------------------------------------------------------------
# 9. Warm classifier cache
# ------------------------------------------------------------

def warm_classifier_cache(data_df):

    """
    Batch-score every combination of granted TOP_K
    permissions for every application.

    The resulting probabilities are stored in _CLS_CACHE.
    """

    vecs = []
    keys = []

    for _, row in data_df.iterrows():

        # ----------------------------------------------------
        # Requested permissions + metadata
        # ----------------------------------------------------

        req = make_requested_vector(row)

        meta = make_metadata_vector(row)

        # ----------------------------------------------------
        # Permissions from TOP_K that this app actually requests
        # ----------------------------------------------------

        req_perms = [
            p
            for p in RL_FEATURES
            if req.get(p, 0) == 1
        ]

        # ----------------------------------------------------
        # Generate every possible granted subset
        # ----------------------------------------------------

        for r in range(
            len(req_perms) + 1
        ):

            for subset in itertools.combinations(
                req_perms,
                r
            ):

                # Granted permission dictionary
                g = {
                    p: (
                        1
                        if p in subset
                        else 0
                    )
                    for p in RL_FEATURES
                }

                # Build classifier input
                v = _feature_vector(
                    g,
                    req,
                    meta
                )

                # Use vector bytes as cache key
                k = v.tobytes()

                # Avoid duplicate prediction
                if k not in _CLS_CACHE:

                    vecs.append(v)
                    keys.append(k)

    # --------------------------------------------------------
    # Batch prediction
    # --------------------------------------------------------

    if vecs:

        X_batch = np.vstack(vecs)

        probs = predict_malicious_proba(
            X_batch
        )

        _CLS_CACHE.update(
            dict(
                zip(
                    keys,
                    map(float, probs)
                )
            )
        )

    return len(vecs)


# ============================================================
# 10. Remove duplicate DataFrame columns temporarily
# ============================================================

# This does NOT modify METADATA_FEATURES.
# It only prevents pandas from returning a Series when
# row[column_name] encounters duplicate column names.

train_df_balanced = train_df_balanced.loc[
    :,
    ~train_df_balanced.columns.duplicated()
]

val_df = val_df.loc[
    :,
    ~val_df.columns.duplicated()
]

test_df = test_df.loc[
    :,
    ~test_df.columns.duplicated()
]


# ============================================================
# 11. Clear old classifier cache
# ============================================================

_CLS_CACHE.clear()


# ============================================================
# 12. Warm classifier cache
# ============================================================

for name, d in [
    ("train_balanced", train_df_balanced),
    ("val", val_df),
    ("test", test_df)
]:

    new_vectors = warm_classifier_cache(d)

    print(
        f"Cached classifier risk for {name}: "
        f"{new_vectors} new vectors"
    )


# ============================================================
# 13. Display normalized individual permission risk
# ============================================================

print(
    "\nNormalized individual risk per TOP_K permission:"
)

for p in RL_FEATURES:

    print(
        f"   {p:20s} -> "
        f"{individual_risk_normalized(p):.4f}"
    )


# ============================================================
# 14. Diagnostic information
# ============================================================

print(
    "\nTotal classifier cache size:",
    len(_CLS_CACHE)
)

print(
    "Number of RL features:",
    len(RL_FEATURES)
)

print(
    "Number of classifier permission features:",
    len(CLF_PERMISSION_FEATURES)
)

print(
    "Number of metadata features:",
    len(METADATA_FEATURES)
)

## 16. RL Environment

Sequential Allow/Deny over the TOP_K permissions in Borda-Garrett rank order. State = `(step, granted_mask)`.
The environment is built from the request vector only — never the `Class` label.

```
ALLOW:  reward = ALPHA*Utility - BETA*(0.5*IndRiskNorm + 0.5*ClassifierRisk) - GAMMA*InteractionRisk
DENY:   reward = -(ALPHA*Utility) * denial_cost      (denial_cost = 0.5)
```

In [ ]:
class PermissionEnv:
    def __init__(self, requested_vector, alpha, beta, gamma, metadata_vector, denial_cost=0.5):
        self.requested, self.metadata = requested_vector, metadata_vector
        self.order = RL_FEATURES
        self.alpha, self.beta, self.gamma, self.denial_cost = alpha, beta, gamma, denial_cost
        self.reset()

    def reset(self):
        self.step_idx, self.done = 0, False
        self.granted = {p: 0 for p in self.order}
        self.decided = set()
        return self._state()

    def _state(self):
        return (self.step_idx, tuple(self.granted[p] for p in self.order))

    def step(self, action):
        perm = self.order[self.step_idx]
        if self.requested.get(perm, 0) == 0:
            self.granted[perm] = 0
            self.decided.add(perm)
            reward = 0.0
        else:
            util = utility_map[perm]
            if action == 1:
                self.granted[perm] = 1
                self.decided.add(perm)
                cls_r = classifier_risk(self.granted, self.requested, self.metadata)
                int_r, _ = interaction_risk(self.granted, self.decided)
                reward = self.alpha * util - self.beta * (0.5 * individual_risk_normalized(perm) + 0.5 * cls_r) - self.gamma * int_r
            else:
                self.granted[perm] = 0
                self.decided.add(perm)
                reward = -(self.alpha * util) * self.denial_cost
        self.step_idx += 1
        self.done = self.step_idx >= len(self.order)
        return self._state(), reward, self.done, {}

# Sanity check: Allow-all must beat Deny-all for a fully-requested probe app
probe_req = {p: 1 for p in PERMISSION_NAMES}
probe_req.update({p: 0 for p in CLF_PERMISSION_FEATURES})
for p in CLF_PERMISSION_FEATURES:
    if RAW_TO_AGG.get(p) in RL_FEATURES:
        probe_req[p] = 1
probe_meta = [float(train_df_balanced[m].mean()) if m in train_df_balanced else float(train_df[m].mean()) for m in METADATA_FEATURES]
totals = {}
for a in (1, 0):
    env = PermissionEnv(probe_req, 0.6, 0.25, 0.15, probe_meta); env.reset(); tot, done = 0.0, False
    while not done:
        _, r, done, _ = env.step(a); tot += r
    totals[a] = tot
print("Allow-everything reward:", round(totals[1], 4), "| Deny-everything reward:", round(totals[0], 4))
# These totals need not have a fixed ordering: classifier and interaction penalties
# may rationally make deny-all preferable for this probe. Check the reward outputs
# and risk bounds instead of imposing an outcome the reward formula does not guarantee.
assert all(np.isfinite(value) for value in totals.values()), "Probe episode produced a non-finite reward."
probe_int_risk, _ = interaction_risk(
    {p: 1 for p in RL_FEATURES}, set(RL_FEATURES)
)
assert 0.0 <= probe_int_risk <= 1.0, "Interaction risk escaped [0, 1]."
print("PASS: probe rewards are finite and interaction risk is bounded in [0, 1].")

# Matching semantics: positives require grants; NOT conditions require an explicit denial/absence decision.
_test_pos = {"Conditions": (("Camera", 1),)}
_test_neg = {"Conditions": (("Camera", 0),)}
assert _rule_matches(_test_pos, {"Camera": 1}, {"Camera"})
assert not _rule_matches(_test_pos, {"Camera": 0}, {"Camera"})
assert not _rule_matches(_test_neg, {"Camera": 0}, set()), "Undecided must not count as denied"
assert _rule_matches(_test_neg, {"Camera": 0}, {"Camera"}), "Final/decided denial should satisfy NOT"
print("PASS: positive, negative, undecided, and completed-decision rule matching checks.")

## 17. Q-Learning Agent (Tabular)

In [ ]:
def train_q_learning(data_df, a_c, b_c, g_c, n_passes=QL_PASSES, lr=QL_ALPHA, discount=QL_GAMMA,
                     eps_start=QL_EPS_START, eps_min=QL_EPS_MIN, eps_decay=QL_EPS_DECAY, seed=RANDOM_STATE):
    rng = np.random.RandomState(seed)
    Q = {}
    rows = data_df.sample(frac=1.0, random_state=seed).reset_index(drop=True)   # shuffle once
    envs_in = [(make_requested_vector(r), make_metadata_vector(r)) for _, r in rows.iterrows()]
    n_rows, n_eps = len(rows), n_passes * len(rows)
    eps, hist = eps_start, []
    for ep in range(n_eps):
        req, meta = envs_in[ep % n_rows]
        env = PermissionEnv(req, a_c, b_c, g_c, meta)
        s = env.reset(); ep_r, n_allow, n_deny, done = 0.0, 0, 0, False
        while not done:
            if rng.rand() < eps:
                a = rng.randint(0, 2)
            else:
                q0, q1 = Q.get((s, 0), 0.0), Q.get((s, 1), 0.0)
                a = int(q1 > q0) if q1 != q0 else rng.randint(0, 2)
            perm = env.order[s[0]]
            s2, r, done, _ = env.step(a)
            if req.get(perm, 0) == 1:
                n_allow += a; n_deny += 1 - a
            best_next = 0.0 if done else max(Q.get((s2, 0), 0.0), Q.get((s2, 1), 0.0))
            Q[(s, a)] = Q.get((s, a), 0.0) + lr * (r + discount * best_next - Q.get((s, a), 0.0))
            ep_r += r; s = s2
        eps = max(eps_min, eps * eps_decay)
        hist.append((ep, ep_r, eps, n_allow, n_deny))
    return Q, pd.DataFrame(hist, columns=["episode", "reward", "epsilon", "n_allow", "n_deny"])

def greedy_action(Q, s):
    return 1 if Q.get((s, 1), 0.0) >= Q.get((s, 0), 0.0) else 0

## 18. RL Training (default coefficients, TRAIN)

In [ ]:
Q_table_default, rl_training_history = train_q_learning(train_df_balanced, 0.60, 0.25, 0.15)
print("Episodes:", len(rl_training_history), "| Q-table entries:", len(Q_table_default))
print("Mean reward, first 500 episodes:", round(rl_training_history["reward"].head(500).mean(), 4))
print("Mean reward, last 500 episodes: ", round(rl_training_history["reward"].tail(500).mean(), 4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(rl_training_history["episode"], rl_training_history["reward"], alpha=0.2, label="episode reward")
axes[0].plot(rl_training_history["episode"], rl_training_history["reward"].rolling(200).mean(), color="crimson", label="200-ep rolling mean")
axes[0].set_title("Q-Learning Learning Curve"); axes[0].set_xlabel("Episode"); axes[0].legend()
axes[1].plot(rl_training_history["episode"], rl_training_history["epsilon"], color="darkorange")
axes[1].set_title("Exploration Rate Decay"); axes[1].set_xlabel("Episode")
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "learning_curve.png"), dpi=120); plt.show()

## 19. Validation-Based Reward-Coefficient + Threshold Selection (precision-constrained)

For each `(ALPHA, BETA, GAMMA)`: train a Q-table on TRAIN, apply it to VALIDATION, then choose the
Low/not-Low threshold as **the most accurate threshold whose bootstrap 10th-percentile precision ≥
`PRECISION_TARGET`** and recall ≥
`MIN_RECALL_FLOOR` (fallback: max F0.5). Candidates are ranked by that constrained validation accuracy,
then precision. A small predefined grid also tunes the individual-risk/classifier-risk score weights while 
keeping the interaction-risk weight fixed; the precision and recall safeguards apply to every candidate. 
**The test set is not used.**

In [ ]:
def find_precision_threshold(scores, y_true, precision_target=PRECISION_TARGET,
                             min_recall=MIN_RECALL_FLOOR, grid=None):
    """Maximize accuracy under precision and recall constraints; fall back to max F0.5.
    The bootstrap lower-bound precision must clear the target, while recall stays above the floor.
    If no threshold satisfies the constraint, fall back to max F0.5.
    """
    if grid is None:
        grid = np.round(np.arange(0.20, 0.951, 0.005), 3)

    scores, y_true = np.asarray(scores), np.asarray(y_true)
    B = _boot_idx(len(y_true))
    yb, sb = y_true[B], scores[B]

    rows = []
    for t in grid:
        pred = (scores > t).astype(int)
        m = classification_metrics(y_true, pred)
        m["F0.5"] = fbeta_score(y_true, pred, beta=FALLBACK_BETA, zero_division=0)
        m["Threshold"] = float(t)

        pb = sb > t
        tp = (pb & (yb == 1)).sum(axis=1)
        npos = pb.sum(axis=1)
        m["PrecisionLB"] = float(np.quantile(
            np.where(npos > 0, tp / np.maximum(npos, 1), 0),
            THRESH_PRECISION_QUANTILE
        ))
        rows.append(m)

    tab = pd.DataFrame(rows)
    ok = tab[(tab["PrecisionLB"] >= precision_target) & (tab["Recall"] >= min_recall)]

    if len(ok):
        best = ok.sort_values(["Accuracy", "Precision", "F0.5"], ascending=False).iloc[0]
        rule = f"max accuracy subject to PrecisionLB >= {precision_target} and Recall >= {min_recall}"
    else:
        best = tab.sort_values("F0.5", ascending=False).iloc[0]
        rule = f"fallback: max F{FALLBACK_BETA}"

    return float(best["Threshold"]), best, rule, tab


In [ ]:
PRIVACY_SCORE_WEIGHTS = {"individual": 0.05, "classifier": 0.90, "interaction": 0.05}

def privacy_class(score, low_thr=PRIVACY_LOW_THRESHOLD, high_thr=PRIVACY_HIGH_THRESHOLD):
    return "Low" if score <= low_thr else ("Medium" if score <= high_thr else "High")

def classification_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
    }

def _score_record(row, requested, granted_dict, ep_reward=np.nan):
    granted_set = frozenset(p for p in RL_FEATURES if granted_dict.get(p, 0) == 1)
    requested_set = frozenset(p for p in RL_FEATURES if requested.get(p, 0) == 1)
    meta = make_metadata_vector(row)
    ind = np.mean([individual_risk_normalized(p) for p in granted_set]) if granted_set else 0.0
    cls_r = classifier_risk(granted_dict, requested, meta)
    int_r, triggered = interaction_risk(granted_dict, RL_FEATURES)
    util = sum(utility_map[p] for p in granted_set) / len(requested_set) if requested_set else 0.0
    privacy_score = float(np.clip(
        PRIVACY_SCORE_WEIGHTS["individual"] * ind
        + PRIVACY_SCORE_WEIGHTS["classifier"] * cls_r
        + PRIVACY_SCORE_WEIGHTS["interaction"] * int_r,
        0.0,
        1.0,
    ))
    return {
        "App": row["App"],
        "TrueLabel": int(row["LabelEncoded"]),
        "RequestedPermissions": ";".join(sorted(requested_set)),
        "AllowedPermissions": ";".join(sorted(granted_set)),
        "DeniedPermissions": ";".join(sorted(requested_set - granted_set)),
        "NumRequested": len(requested_set),
        "NumAllowed": len(granted_set),
        "NumDenied": len(requested_set - granted_set),
        "IndividualRisk": ind,
        "MaliciousProbability": cls_r,
        "InteractionRisk": int_r,
        "TriggeredRules": "; ".join(triggered),
        "UtilityScore": util,
        "PrivacyScore": privacy_score,
        "RLReward": ep_reward,
    }

def apply_policy(Q, data_df, a_c, b_c, g_c):
    records = []
    for _, row in data_df.iterrows():
        requested, metadata = make_requested_vector(row), make_metadata_vector(row)
        env = PermissionEnv(requested, a_c, b_c, g_c, metadata)
        state = env.reset()
        episode_reward, done = 0.0, False
        while not done:
            state, reward, done, _ = env.step(greedy_action(Q, state))
            episode_reward += reward
        records.append(_score_record(row, requested, env.granted, episode_reward))
    return pd.DataFrame(records)

_BOOT_IDX_CACHE = {}
def _boot_idx(n):
    if n not in _BOOT_IDX_CACHE:
        _BOOT_IDX_CACHE[n] = np.random.RandomState(RANDOM_STATE).randint(
            0, n, (THRESH_N_BOOT, n)
        )
    return _BOOT_IDX_CACHE[n]

selection_records, trained_qtables, val_results_cache = [], {}, {}
interaction_weight = 0.05
individual_weight_candidates = [0.05, 0.15, 0.30]
for alpha, beta, gamma in COEF_CANDIDATES:
    q_table, history = train_q_learning(train_df_balanced, alpha, beta, gamma)
    trained_qtables[(alpha, beta, gamma)] = (q_table, history)
    validation_results = apply_policy(q_table, val_df, alpha, beta, gamma)
    val_results_cache[(alpha, beta, gamma)] = validation_results

    for individual_weight in individual_weight_candidates:
        classifier_weight = 1.0 - interaction_weight - individual_weight
        validation_scores = np.clip(
            individual_weight * validation_results["IndividualRisk"].values
            + classifier_weight * validation_results["MaliciousProbability"].values
            + interaction_weight * validation_results["InteractionRisk"].values,
            0.0,
            1.0,
        )
        threshold, best, rule, _ = find_precision_threshold(
            validation_scores,
            validation_results["TrueLabel"].values,
        )
        selection_records.append({
            "Alpha": alpha,
            "Beta": beta,
            "Gamma": gamma,
            "Threshold": threshold,
            "Rule": rule,
            "IndividualWeight": individual_weight,
            "ClassifierWeight": classifier_weight,
            "InteractionWeight": interaction_weight,
            "Val_Accuracy": best["Accuracy"],
            "Val_Precision": best["Precision"],
            "Val_Recall": best["Recall"],
            "Val_F1": best["F1"],
            "Val_F0.5": best["F0.5"],
            "Val_PrecisionLB": best["PrecisionLB"],
            "Val_AllowRate": validation_results["NumAllowed"].sum()
            / max(validation_results["NumRequested"].sum(), 1),
            "Val_AvgUtility": validation_results["UtilityScore"].mean(),
            "Val_AvgPrivacyRisk": float(validation_scores.mean()),
        })

selection_df = (
    pd.DataFrame(selection_records)
    .assign(_meets=lambda frame: frame["Val_PrecisionLB"] >= PRECISION_TARGET)
    .sort_values(
        ["_meets", "Val_Accuracy", "Val_Precision", "Val_AvgUtility", "Val_AvgPrivacyRisk"],
        ascending=[False, False, False, False, True],
    )
    .drop(columns="_meets")
    .reset_index(drop=True)
)
display(selection_df)

BEST_ALPHA, BEST_BETA, BEST_GAMMA, BEST_THRESHOLD = selection_df.iloc[0][
    ["Alpha", "Beta", "Gamma", "Threshold"]
]
PRIVACY_SCORE_WEIGHTS = {
    "individual": float(selection_df.iloc[0]["IndividualWeight"]),
    "classifier": float(selection_df.iloc[0]["ClassifierWeight"]),
    "interaction": float(selection_df.iloc[0]["InteractionWeight"]),
}
BEST_Q, BEST_HISTORY = trained_qtables[(BEST_ALPHA, BEST_BETA, BEST_GAMMA)]
best_val_results = val_results_cache[(BEST_ALPHA, BEST_BETA, BEST_GAMMA)]
best_val_results["PrivacyScore"] = np.clip(
    PRIVACY_SCORE_WEIGHTS["individual"] * best_val_results["IndividualRisk"].values
    + PRIVACY_SCORE_WEIGHTS["classifier"] * best_val_results["MaliciousProbability"].values
    + PRIVACY_SCORE_WEIGHTS["interaction"] * best_val_results["InteractionRisk"].values,
    0.0,
    1.0,
)
print(
    f"Selected: ALPHA={BEST_ALPHA}, BETA={BEST_BETA}, GAMMA={BEST_GAMMA}, "
    f"THRESHOLD={BEST_THRESHOLD:.3f} "
    f"(validation accuracy={selection_df.iloc[0]['Val_Accuracy']:.4f}, "
    f"precision={selection_df.iloc[0]['Val_Precision']:.4f})"
)

In [ ]:
# Precision / recall / accuracy vs threshold on VALIDATION for the selected coefficients
_, _, _, thr_tab = find_precision_threshold(
    val_results_cache[(BEST_ALPHA, BEST_BETA, BEST_GAMMA)]["PrivacyScore"].values,
    val_results_cache[(BEST_ALPHA, BEST_BETA, BEST_GAMMA)]["TrueLabel"].values)
fig, ax = plt.subplots(figsize=(8, 4.5))
for c, col in [("Precision", "firebrick"), ("Recall", "steelblue"), ("Accuracy", "black")]:
    ax.plot(thr_tab["Threshold"], thr_tab[c], label=c, color=col)
ax.axhline(PRECISION_TARGET, ls=":", color="firebrick", lw=1)
ax.axvline(BEST_THRESHOLD, ls="--", color="grey", lw=1, label=f"chosen = {BEST_THRESHOLD:.3f}")
ax.set_xlabel("Privacy-score threshold"); ax.set_title("VALIDATION trade-off (selected coefficients)")
ax.legend(); plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "val_threshold_tradeoff.png"), dpi=120); plt.show()

## 20. Final Test Evaluation (HELD-OUT TEST SET — first and only use)

In [ ]:
rl_test_results = apply_policy(BEST_Q, test_df, BEST_ALPHA, BEST_BETA, BEST_GAMMA)
rl_test_results["PrivacyClass"] = rl_test_results["PrivacyScore"].apply(privacy_class)
y_test_true = rl_test_results["TrueLabel"].values
rl_pred = (rl_test_results["PrivacyScore"].values > BEST_THRESHOLD).astype(int)

rl_metrics = classification_metrics(y_test_true, rl_pred)
rl_metrics["F0.5"] = fbeta_score(y_test_true, rl_pred, beta=0.5)
rl_metrics["ROC_AUC"] = roc_auc_score(y_test_true, rl_test_results["PrivacyScore"])
rl_metrics["AvgPrecision"] = average_precision_score(y_test_true, rl_test_results["PrivacyScore"])
rl_cm = confusion_matrix(y_test_true, rl_pred)

print(f"Validation-tuned threshold: {BEST_THRESHOLD:.3f}")
print("=== RL POLICY -- FINAL TEST METRICS ===")
for k, v in rl_metrics.items():
    print(f"  {k:12s}: {v:.4f}")
print("\nConfusion matrix (rows = true [Benign, Malicious], cols = predicted):")
print(rl_cm)
print()
print(classification_report(y_test_true, rl_pred, target_names=["Benign", "Malicious"], digits=4))

## 20b. Test-Set Uncertainty (Bootstrap 95% CIs)

In [ ]:
n_boot = 2000
rng_boot = np.random.RandomState(RANDOM_STATE)
n_test = len(y_test_true)
boot = {"Accuracy": [], "Precision": [], "Recall": [], "F1": []}
for _ in range(n_boot):
    i = rng_boot.randint(0, n_test, n_test)
    m = classification_metrics(y_test_true[i], rl_pred[i])
    for k in boot:
        boot[k].append(m[k])
ci_df = pd.DataFrame({k: {"Point": rl_metrics[k], "CI_2.5%": np.percentile(v, 2.5), "CI_97.5%": np.percentile(v, 97.5)}
                      for k, v in boot.items()}).T
print(f"Bootstrap over {n_test} test apps, {n_boot} resamples:")
ci_df.round(4)

In [ ]:
rl_classification_metrics_df = pd.DataFrame([{"Model": "RL_Policy", **rl_metrics,
    "AllowRate": rl_test_results["NumAllowed"].sum() / max(rl_test_results["NumRequested"].sum(), 1)}])
rl_privacy_utility_metrics_df = pd.DataFrame([{
    "AvgPrivacyScore": rl_test_results["PrivacyScore"].mean(),
    "AvgPrivacyRisk_Individual": rl_test_results["IndividualRisk"].mean(),
    "AvgMaliciousProbability": rl_test_results["MaliciousProbability"].mean(),
    "AvgInteractionRisk": rl_test_results["InteractionRisk"].mean(),
    "AvgUtility": rl_test_results["UtilityScore"].mean(),
    "AvgPermissionsRequested": rl_test_results["NumRequested"].mean(),
    "AvgPermissionsAllowed": rl_test_results["NumAllowed"].mean(),
    "AvgPermissionsDenied": rl_test_results["NumDenied"].mean(),
    "PermissionAllowRate": rl_test_results["NumAllowed"].sum() / max(rl_test_results["NumRequested"].sum(), 1),
    "PrivacyUtilityScore": rl_test_results["UtilityScore"].mean() - rl_test_results["PrivacyScore"].mean(),
}])
display(rl_classification_metrics_df.round(4))
display(rl_privacy_utility_metrics_df.round(4))

## 22. Privacy-Utility Analysis

In [ ]:
print("Privacy class distribution on TEST (fixed 0.33 / 0.67 bands):")
print(rl_test_results["PrivacyClass"].value_counts())
print("\nMean PrivacyScore  Benign:", round(rl_test_results.loc[rl_test_results.TrueLabel == 0, "PrivacyScore"].mean(), 4),
      "| Malicious:", round(rl_test_results.loc[rl_test_results.TrueLabel == 1, "PrivacyScore"].mean(), 4))

test_with_genre = rl_test_results.assign(Genre=test_df["Category"].values, Pred=rl_pred)
per_genre = test_with_genre.groupby("Genre").apply(
    lambda d: pd.Series({"N": len(d), "Accuracy": accuracy_score(d.TrueLabel, d.Pred),
                         "Precision": precision_score(d.TrueLabel, d.Pred, zero_division=0),
                         "Recall": recall_score(d.TrueLabel, d.Pred, zero_division=0)}), include_groups=False)
print("\nPer-genre TEST metrics (informational):")
per_genre.round(4)